# GovOn-EXAONE LoRA v2 머지 & AWQ 양자화 파이프라인

이 노트북은 LoRA v2 어댑터를 베이스 모델에 머지한 뒤 AWQ 양자화를 수행한다.

## 파이프라인 개요

1. **환경 설정** - 호환 버전 패키지 설치
2. **API 키 설정** - WandB & HuggingFace
3. **LoRA 머지** - 베이스 모델 + LoRA v2 어댑터 → BF16 머지 모델
4. **머지 검증** - 추론 테스트로 머지 정상 확인
5. **AWQ 양자화** - W4A16g128 양자화 적용
6. **양자화 검증** - AWQ 모델 추론 테스트
7. **HuggingFace 업로드** - 머지/AWQ 모델 Hub에 업로드
8. **WandB 로깅** - 전체 결과 기록

## 모델 정보

| 항목 | 값 |
|------|----|
| 베이스 모델 | `LGAI-EXAONE/EXAONE-Deep-7.8B` |
| LoRA 어댑터 | `umyunsang/GovOn-EXAONE-LoRA-v2` |
| 머지 출력 | `umyunsang/GovOn-EXAONE-Merged-v2` |
| AWQ 출력 | `umyunsang/GovOn-EXAONE-AWQ-v2` |
| 양자화 설정 | W4A16, group_size=128, zero_point=True |

## 요구사항

- Google Colab GPU 런타임 (T4/L4/A100 권장)
- HuggingFace 토큰 (write 권한)
- WandB API 키

---
## 1. API 키 설정

WandB와 HuggingFace API 키를 입력한다. 이 셀을 먼저 실행하여 키를 설정해야 한다.

In [ ]:
# ============================================================
# API 키 설정 (이 셀을 먼저 실행)
# ============================================================
import os
from getpass import getpass

# WandB API 키
WANDB_API_KEY = getpass("WandB API Key: ")
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

# HuggingFace 토큰 (write 권한 필요)
HF_TOKEN = getpass("HuggingFace Token (write 권한): ")
os.environ["HF_TOKEN"] = HF_TOKEN

print("API 키 설정 완료")

---
## 2. 환경 설정

EXAONE 호환 버전으로 패키지를 설치한다.

**중요**: LoRA 학습 시점의 transformers 버전(4.44~4.49)과 EXAONE 모델 코드 revision을 맞춰야 한다.  
최신 transformers(v5+)에서는 `modeling_exaone.py`의 forward pass 구조가 변경되어 LoRA 가중치가 올바르게 적용되지 않는다.

In [ ]:
# ============================================================
# 패키지 설치 (EXAONE 호환 버전 고정)
# ============================================================
# 기존 버전 충돌 방지를 위해 제거 후 재설치
!pip uninstall -y transformers accelerate autoawq -q

# 학습 시점 호환 버전 설치
!pip install -q \
    "transformers>=4.44,<4.50" \
    "accelerate>=1.3.0,<2.0" \
    "peft>=0.18.0" \
    "bitsandbytes" \
    "torch>=2.1.0"

# AWQ 양자화 라이브러리
!pip install -q autoawq

# 유틸리티
!pip install -q \
    wandb \
    huggingface_hub \
    "protobuf<5.0.0"

print("\n" + "=" * 60)
print("설치 완료. 버전 확인:")
print("=" * 60)

import transformers, peft, accelerate, torch
print(f"  transformers: {transformers.__version__}")
print(f"  peft:         {peft.__version__}")
print(f"  accelerate:   {accelerate.__version__}")
print(f"  torch:        {torch.__version__}")
print(f"  CUDA:         {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

try:
    import awq
    print(f"  autoawq:      {awq.__version__}")
except ImportError:
    print("  autoawq:      import 실패 - 커널 재시작 필요할 수 있음")

# transformers 버전 검증
tv = transformers.__version__
major, minor = int(tv.split('.')[0]), int(tv.split('.')[1])
if not (major == 4 and 44 <= minor <= 49):
    print(f"\n⚠️  경고: transformers {tv}는 호환 범위(4.44~4.49)를 벗어남!")
    print("    LoRA 어댑터 로드 시 문제가 발생할 수 있다.")
else:
    print(f"\n✅ transformers {tv} - 호환 범위 내")

---
## 3. 설정 및 임포트

In [ ]:
# ============================================================
# 설정 및 임포트
# ============================================================
import os
import sys
import json
import time
import gc
import re
import torch
import wandb
from datetime import datetime
from huggingface_hub import HfApi, hf_hub_download

# ── 모델 설정 ──
BASE_MODEL_ID = "LGAI-EXAONE/EXAONE-Deep-7.8B"
ADAPTER_ID = "umyunsang/GovOn-EXAONE-LoRA-v2"
EXAONE_REVISION = "17b70148e344"  # 학습 호환 마지막 revision (2025-03-19)

# ── 출력 경로 ──
MERGED_OUTPUT_DIR = "/content/models/merged_model"
AWQ_OUTPUT_DIR = "/content/models/awq_quantized_model"

# ── HuggingFace Hub 리포 이름 ──
HF_MERGED_REPO = "umyunsang/GovOn-EXAONE-Merged-v2"
HF_AWQ_REPO = "umyunsang/GovOn-EXAONE-AWQ-v2"

# ── AWQ 양자화 설정 ──
AWQ_QUANT_CONFIG = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

# ── 캘리브레이션 설정 ──
CALIB_N_SAMPLES = 512
CALIB_SEED = 42

# ── WandB 설정 ──
WANDB_PROJECT = "exaone-civil-complaint"

print("설정 완료:")
print(f"  베이스 모델:    {BASE_MODEL_ID}")
print(f"  LoRA 어댑터:    {ADAPTER_ID}")
print(f"  EXAONE 리비전:  {EXAONE_REVISION}")
print(f"  머지 출력:      {MERGED_OUTPUT_DIR}")
print(f"  AWQ 출력:       {AWQ_OUTPUT_DIR}")
print(f"  HF 머지 리포:   {HF_MERGED_REPO}")
print(f"  HF AWQ 리포:    {HF_AWQ_REPO}")
print(f"  AWQ 설정:       {AWQ_QUANT_CONFIG}")

---
## 4. Stage 1: LoRA 어댑터 머지

베이스 모델(BF16)에 LoRA v2 어댑터를 머지하여 풀 모델을 생성한다.

### 주의사항
- `revision` 파라미터로 학습 시점의 EXAONE 모델 코드를 고정한다
- 머지 후 파라미터 수가 베이스 모델과 동일한지 검증한다
- LoRA 모듈이 완전히 제거되었는지 확인한다

In [ ]:
# ============================================================
# Stage 1: LoRA 어댑터 머지
# ============================================================
merge_start_time = time.time()

# WandB 초기화
run = wandb.init(
    project=WANDB_PROJECT,
    name=f"merge-quantize-v2-{datetime.now().strftime('%Y%m%d-%H%M')}",
    tags=["merge", "awq", "lora-v2", "exaone-7.8b", "pipeline"],
    config={
        "base_model": BASE_MODEL_ID,
        "adapter": ADAPTER_ID,
        "exaone_revision": EXAONE_REVISION,
        "merged_output_dir": MERGED_OUTPUT_DIR,
        "awq_output_dir": AWQ_OUTPUT_DIR,
        "awq_quant_config": AWQ_QUANT_CONFIG,
        "calib_samples": CALIB_N_SAMPLES,
        "stage": "merge_and_quantize_v2",
    }
)

print("=" * 60)
print("Stage 1: LoRA v2 어댑터 머지")
print("=" * 60)

# ── Step 1: 어댑터 설정 검증 ──
print("\n[1/6] 어댑터 설정 검증...")
config_path = hf_hub_download(ADAPTER_ID, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

print(f"  베이스 모델:    {adapter_config['base_model_name_or_path']}")
print(f"  PEFT 타입:      {adapter_config['peft_type']}")
print(f"  Rank (r):       {adapter_config['r']}")
print(f"  Alpha:          {adapter_config['lora_alpha']}")
print(f"  Target modules: {adapter_config['target_modules']}")
print(f"  Dropout:        {adapter_config['lora_dropout']}")

assert adapter_config['base_model_name_or_path'] == BASE_MODEL_ID, \
    f"베이스 모델 불일치: {adapter_config['base_model_name_or_path']} != {BASE_MODEL_ID}"
assert adapter_config['peft_type'] == 'LORA', "LORA 타입이 아님"

wandb.log({"adapter_config": adapter_config})
print("  [OK] 어댑터 설정 검증 완료")

# ── Step 2: 토크나이저 로드 ──
print("\n[2/6] 토크나이저 로드...")
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    revision=EXAONE_REVISION,
)
print(f"  Vocab size:        {tokenizer.vocab_size}")
print(f"  Model max length:  {tokenizer.model_max_length}")
print(f"  EOS token:         {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# ── Step 3: 베이스 모델 로드 (BF16) ──
print("\n[3/6] 베이스 모델 로드 (BF16)...")
from transformers import AutoModelForCausalLM

mem_before = torch.cuda.memory_allocated() / 1024**3
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    revision=EXAONE_REVISION,
)
mem_after_base = torch.cuda.memory_allocated() / 1024**3

base_param_count = sum(p.numel() for p in base_model.parameters())
print(f"  베이스 모델 파라미터: {base_param_count:,}")
print(f"  GPU 메모리 사용:     {mem_after_base:.2f} GB")

wandb.log({
    "base_model_params": base_param_count,
    "gpu_mem_base_model_gb": mem_after_base,
})

# ── Step 4: LoRA 어댑터 로드 및 머지 ──
print("\n[4/6] LoRA v2 어댑터 로드 및 머지...")
from peft import PeftModel

model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
mem_after_adapter = torch.cuda.memory_allocated() / 1024**3

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  전체 파라미터 (어댑터 포함): {total_params:,}")
print(f"  학습 가능 파라미터 (어댑터): {trainable_params:,}")
print(f"  학습 가능 비율:              {100 * trainable_params / total_params:.4f}%")
print(f"  GPU 메모리 (어댑터 포함):    {mem_after_adapter:.2f} GB")

print("\n  어댑터 가중치를 베이스 모델에 머지 중...")
model = model.merge_and_unload()
mem_after_merge = torch.cuda.memory_allocated() / 1024**3

merged_param_count = sum(p.numel() for p in model.parameters())
print(f"  머지 후 파라미터: {merged_param_count:,}")
print(f"  GPU 메모리:       {mem_after_merge:.2f} GB")

# 검증: 파라미터 수 일치 확인
assert merged_param_count == base_param_count, \
    f"파라미터 수 불일치: {merged_param_count} != {base_param_count}"
print("  [OK] 파라미터 수 베이스 모델과 일치")

# 검증: LoRA 모듈 잔존 확인
has_lora = any("lora" in name.lower() for name, _ in model.named_parameters())
assert not has_lora, "머지 후에도 LoRA 모듈이 남아있음!"
print("  [OK] LoRA 모듈 완전 제거 확인")

wandb.log({
    "adapter_trainable_params": trainable_params,
    "adapter_trainable_pct": 100 * trainable_params / total_params,
    "merged_param_count": merged_param_count,
    "gpu_mem_after_merge_gb": mem_after_merge,
    "merge_param_count_match": merged_param_count == base_param_count,
})

# ── Step 5: 머지 모델 저장 ──
print(f"\n[5/6] 머지 모델 저장: {MERGED_OUTPUT_DIR}")
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)
model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

# 모델 크기 계산
merged_size_bytes = sum(
    os.path.getsize(os.path.join(MERGED_OUTPUT_DIR, f))
    for f in os.listdir(MERGED_OUTPUT_DIR)
    if f.endswith(('.safetensors', '.bin'))
)
merged_size_gb = merged_size_bytes / 1024**3
print(f"  디스크 크기: {merged_size_gb:.2f} GB")

# ── Step 6: 머지 모델 추론 테스트 ──
print("\n[6/6] 머지 모델 추론 테스트...")

def get_eos_ids(tok):
    """EXAONE EOS 토큰 ID 목록 반환"""
    ids = [tok.eos_token_id]
    for special_tok in ['[|endofturn|]', '<|endofturn|>']:
        tid = tok.convert_tokens_to_ids(special_tok)
        if tid is not None and tid != tok.unk_token_id and tid not in ids:
            ids.append(tid)
    return ids

test_prompts = [
    "주민등록증 재발급 절차가 어떻게 되나요?",
    "우리 동네 도로에 포트홀이 생겨서 위험합니다.",
]

eos_ids = get_eos_ids(tokenizer)
print(f"  EOS 토큰 IDs: {eos_ids}")

for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."},
        {"role": "user", "content": f"다음 민원에 대해 공손하고 명확한 답변을 작성하세요.\n\n민원 내용: {prompt}"},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos_ids,
        )

    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=False)
    # <thought> 태그 제거
    clean_response = re.sub(r"<thought>.*?</thought>", "", response, flags=re.DOTALL).strip()
    
    print(f"\n  --- 테스트 {i+1} ---")
    print(f"  질의: {prompt}")
    print(f"  응답 (300자): {clean_response[:300]}")
    has_eos = any(tok in response for tok in ['[|endofturn|]', '<|endofturn|>'])
    print(f"  EOS 생성: {'Yes' if has_eos else 'No'}")

merge_elapsed = time.time() - merge_start_time

wandb.log({
    "merged_model_size_gb": merged_size_gb,
    "merge_time_seconds": merge_elapsed,
})

# 머지 로그 저장
merge_log = {
    "stage": "1_lora_merge_v2",
    "timestamp": datetime.now().isoformat(),
    "base_model": BASE_MODEL_ID,
    "adapter": ADAPTER_ID,
    "exaone_revision": EXAONE_REVISION,
    "base_param_count": base_param_count,
    "adapter_trainable_params": trainable_params,
    "merged_param_count": merged_param_count,
    "model_size_gb": merged_size_gb,
    "elapsed_seconds": merge_elapsed,
    "adapter_config": adapter_config,
}
with open(os.path.join(MERGED_OUTPUT_DIR, "merge_log.json"), "w") as f:
    json.dump(merge_log, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{'=' * 60}")
print(f"Stage 1 완료! 소요 시간: {merge_elapsed:.1f}초 ({merge_elapsed/60:.1f}분)")
print(f"머지 모델 저장 위치: {MERGED_OUTPUT_DIR}")
print(f"{'=' * 60}")

# 메모리 정리 (AWQ에서 다시 로드하므로 여기서 해제)
del model, base_model
gc.collect()
torch.cuda.empty_cache()
print("\nGPU 메모리 해제 완료")

---
## 5. Stage 2: AWQ 양자화

머지된 BF16 모델에 AWQ(Activation-aware Weight Quantization)를 적용한다.

- **양자화 방식**: W4A16g128 (4bit 가중치, 16bit 활성화, group_size=128)
- **캘리브레이션**: 민원 도메인 데이터 512개 샘플 사용
- **예상 크기**: BF16 ~15GB → AWQ ~4.5GB (약 3.3x 압축)

In [ ]:
# ============================================================
# Stage 2: AWQ 양자화
# ============================================================
import random

quant_start_time = time.time()

print("=" * 60)
print("Stage 2: AWQ 양자화 (W4A16g128)")
print("=" * 60)

# ── Step 1: 캘리브레이션 데이터 준비 ──
print("\n[1/4] 캘리브레이션 데이터 준비...")

# 토크나이저 재로드 (머지 모델에서)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MERGED_OUTPUT_DIR, trust_remote_code=True)

def prepare_calibration_data_inline(n_samples=512, seed=42):
    """
    민원 도메인 캘리브레이션 데이터를 생성한다.
    
    Colab에서는 로컬 데이터 파일이 없을 수 있으므로,
    HuggingFace에서 다운로드하거나 인라인으로 생성한다.
    """
    random.seed(seed)
    
    # 방법 1: 로컬 데이터 파일이 있는 경우
    local_data_paths = [
        "/content/ondevice-ai-civil-complaint/data/processed/v2_train.jsonl",
        "/content/data/v2_train.jsonl",
    ]
    
    for data_path in local_data_paths:
        if os.path.exists(data_path):
            print(f"  로컬 데이터 사용: {data_path}")
            samples = []
            with open(data_path, 'r') as f:
                lines = f.readlines()
            random.shuffle(lines)
            
            for line in lines[:n_samples * 2]:
                try:
                    item = json.loads(line.strip())
                    text = item.get('text', '')
                    if len(text) > 100:
                        samples.append(text)
                except (json.JSONDecodeError, KeyError):
                    continue
                if len(samples) >= n_samples:
                    break
            
            if len(samples) >= n_samples // 2:
                print(f"  캘리브레이션 샘플 {len(samples)}개 준비 완료")
                return samples
    
    # 방법 2: 인라인 민원 도메인 데이터 생성
    print("  로컬 데이터 없음 → 인라인 민원 도메인 데이터 생성")
    
    civil_complaint_examples = [
        # 행정
        ("주민등록증 재발급 절차가 어떻게 되나요?", 
         "주민등록증 재발급은 주민센터(읍면동사무소)를 방문하시거나 정부24 온라인을 통해 신청하실 수 있습니다. 방문 시 본인 확인을 위한 신분증(운전면허증 등)이 필요하며, 수수료는 5,000원입니다. 처리 기간은 약 3~5일 정도 소요됩니다."),
        ("전입신고는 어디서 하나요?",
         "전입신고는 새로운 거주지의 주민센터(읍면동사무소)를 방문하시거나, 정부24(www.gov.kr) 온라인으로 신청하실 수 있습니다. 전입일로부터 14일 이내에 신고하셔야 하며, 세대주 또는 세대원이 신고할 수 있습니다."),
        ("인감증명서 발급에 필요한 서류는 무엇인가요?",
         "인감증명서 발급을 위해서는 본인이 직접 주민센터를 방문하셔야 합니다. 본인 확인을 위한 신분증(주민등록증, 운전면허증 등)을 지참하시면 됩니다. 대리인 발급 시에는 위임장과 대리인 신분증이 추가로 필요합니다."),
        # 교통
        ("도로에 포트홀이 생겨서 위험합니다.",
         "도로 포트홀 신고 감사합니다. 해당 위치를 확인하여 긴급 보수 조치하겠습니다. 정확한 위치(도로명, 인근 건물 등)를 알려주시면 신속하게 처리할 수 있습니다. 긴급한 경우 도로관리과로 직접 연락 부탁드립니다."),
        ("신호등이 고장났습니다.",
         "신호등 고장 신고 감사합니다. 교통안전시설 관리 부서에 즉시 전달하여 점검 및 수리 조치하겠습니다. 해당 교차로의 정확한 위치를 알려주시면 더욱 신속한 처리가 가능합니다."),
        ("버스 노선 변경 요청을 하고 싶습니다.",
         "버스 노선 변경 건의를 접수하겠습니다. 노선 변경은 교통수요 분석, 주민 의견 수렴, 관계기관 협의 등의 절차가 필요하며, 검토 후 결과를 안내드리겠습니다."),
        # 환경
        ("불법 쓰레기 투기를 신고합니다.",
         "불법 쓰레기 투기 신고 감사합니다. 환경관리과에서 현장 확인 후 수거 조치하겠습니다. CCTV 확인 등을 통해 투기자를 확인할 경우 과태료가 부과됩니다. 정확한 투기 장소와 시간을 알려주시면 단속에 도움이 됩니다."),
        ("소음 민원을 접수하고 싶습니다.",
         "소음 민원 접수 감사합니다. 소음 유형(공사장, 생활소음, 교통소음 등)에 따라 담당 부서가 달라집니다. 소음 발생 장소, 시간대, 소음 유형을 상세히 알려주시면 신속하게 조치하겠습니다."),
        # 복지
        ("기초생활수급자 신청 방법을 알려주세요.",
         "기초생활수급자 신청은 주소지 주민센터(읍면동사무소)를 방문하여 신청하실 수 있습니다. 신청 시 소득·재산 조사를 거치며, 가구의 소득인정액이 기준 중위소득의 일정 비율 이하인 경우 수급자로 선정됩니다."),
        ("장애인 등록 절차가 궁금합니다.",
         "장애인 등록은 주소지 주민센터(읍면동사무소)에서 신청하실 수 있습니다. 신청 후 국민연금공단에서 장애 정도를 심사하며, 심사 결과에 따라 장애인으로 등록됩니다."),
        # 문화
        ("지역 문화센터 프로그램 안내 부탁드립니다.",
         "지역 문화센터에서는 다양한 교육 및 문화 프로그램을 운영하고 있습니다. 구체적인 프로그램 일정과 수강 신청은 해당 문화센터 홈페이지나 전화로 확인하실 수 있습니다."),
        # 경제
        ("소상공인 지원금 신청 방법을 알려주세요.",
         "소상공인 지원금은 소상공인시장진흥공단 또는 지자체 경제진흥과를 통해 신청하실 수 있습니다. 지원 대상, 신청 기간, 필요 서류 등은 사업별로 상이하므로 해당 기관에 문의하시기 바랍니다."),
        # 안전
        ("가로등이 꺼져서 밤에 위험합니다.",
         "가로등 고장 신고 감사합니다. 시설관리과에서 현장 확인 후 수리 조치하겠습니다. 정확한 위치(가로등 번호, 도로명 등)를 알려주시면 신속한 처리가 가능합니다."),
        ("재난 대피소 위치를 알고 싶습니다.",
         "재난 대피소는 국민재난안전포털(www.safekorea.go.kr)에서 검색하실 수 있습니다. 주소 또는 현재 위치 기반으로 가까운 대피소를 확인할 수 있으며, 안전디딤돌 앱에서도 조회 가능합니다."),
        # 세금
        ("재산세 납부 기한이 언제인가요?",
         "재산세는 매년 7월과 9월에 나누어 납부합니다. 7월에는 건축물분과 주택분(1/2), 9월에는 토지분과 주택분(나머지 1/2)이 부과됩니다. 납부 기한은 각각 7월 31일, 9월 30일까지입니다."),
        ("자동차세 연납 신청을 하고 싶습니다.",
         "자동차세 연납 신청은 위택스(www.wetax.go.kr) 또는 관할 구청 세무과에서 가능합니다. 1월에 연납 신청 시 약 9.15%의 세액 공제 혜택을 받으실 수 있습니다."),
    ]
    
    samples = []
    system_msg = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."
    
    # 기본 예제로 반복하여 n_samples 만큼 생성
    for idx in range(n_samples):
        q, a = civil_complaint_examples[idx % len(civil_complaint_examples)]
        # EXAONE 챗 템플릿 형식으로 구성
        text = f"[|system|]{system_msg}[|endofturn|]\n[|user|]다음 민원에 대해 공손하고 명확한 답변을 작성하세요.\n\n민원 내용: {q}[|endofturn|]\n[|assistant|]{a}[|endofturn|]"
        samples.append(text)
    
    random.shuffle(samples)
    print(f"  인라인 캘리브레이션 샘플 {len(samples)}개 생성 완료")
    return samples

calib_data = prepare_calibration_data_inline(n_samples=CALIB_N_SAMPLES, seed=CALIB_SEED)
wandb.log({"calibration_samples": len(calib_data)})

# ── Step 2: AutoAWQ로 모델 로드 ──
print("\n[2/4] AutoAWQ로 머지 모델 로드...")
from awq import AutoAWQForCausalLM

model = AutoAWQForCausalLM.from_pretrained(
    MERGED_OUTPUT_DIR,
    safetensors=True,
    trust_remote_code=True,
)
print("  모델 로드 완료")

mem_before_quant = torch.cuda.memory_allocated() / 1024**3
print(f"  GPU 메모리 (양자화 전): {mem_before_quant:.2f} GB")

# ── Step 3: AWQ 양자화 수행 ──
print("\n[3/4] AWQ 양자화 수행...")
print(f"  양자화 설정: {AWQ_QUANT_CONFIG}")
print(f"  캘리브레이션 샘플: {len(calib_data)}개")
print(f"  예상 소요 시간: 20~40분 (GPU에 따라 다름)")

quant_actual_start = time.time()
model.quantize(
    tokenizer,
    quant_config=AWQ_QUANT_CONFIG,
    calib_data=calib_data,
)
quant_elapsed = time.time() - quant_actual_start
print(f"  양자화 완료! 소요 시간: {quant_elapsed:.1f}초 ({quant_elapsed/60:.1f}분)")

# ── Step 4: 양자화 모델 저장 ──
print(f"\n[4/4] 양자화 모델 저장: {AWQ_OUTPUT_DIR}")
os.makedirs(AWQ_OUTPUT_DIR, exist_ok=True)
model.save_quantized(AWQ_OUTPUT_DIR, safetensors=True)
tokenizer.save_pretrained(AWQ_OUTPUT_DIR)

# modeling_exaone.py, configuration_exaone.py 복사 (커스텀 코드)
import shutil
for fname in ["modeling_exaone.py", "configuration_exaone.py"]:
    src = os.path.join(MERGED_OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(AWQ_OUTPUT_DIR, fname))
        print(f"  {fname} 복사 완료")

# 모델 크기 비교
awq_size_bytes = sum(
    os.path.getsize(os.path.join(AWQ_OUTPUT_DIR, f))
    for f in os.listdir(AWQ_OUTPUT_DIR)
    if f.endswith(('.safetensors', '.bin'))
)
awq_size_gb = awq_size_bytes / 1024**3
compression_ratio = merged_size_gb / awq_size_gb if awq_size_gb > 0 else 0
size_reduction_pct = (1 - awq_size_gb / merged_size_gb) * 100 if merged_size_gb > 0 else 0

print(f"\n  크기 비교:")
print(f"    머지 모델 (BF16): {merged_size_gb:.2f} GB")
print(f"    AWQ 모델 (W4):    {awq_size_gb:.2f} GB")
print(f"    압축률:           {compression_ratio:.2f}x")
print(f"    크기 감소:        {size_reduction_pct:.1f}%")

wandb.log({
    "quantization_time_seconds": quant_elapsed,
    "awq_model_size_gb": awq_size_gb,
    "merged_model_size_gb": merged_size_gb,
    "compression_ratio": compression_ratio,
    "size_reduction_pct": size_reduction_pct,
})

# 양자화 로그 저장
quant_log = {
    "stage": "2_awq_quantization_v2",
    "timestamp": datetime.now().isoformat(),
    "merged_model_dir": MERGED_OUTPUT_DIR,
    "output_dir": AWQ_OUTPUT_DIR,
    "quant_config": AWQ_QUANT_CONFIG,
    "calibration_samples": len(calib_data),
    "awq_model_size_gb": awq_size_gb,
    "merged_model_size_gb": merged_size_gb,
    "compression_ratio": compression_ratio,
    "size_reduction_pct": size_reduction_pct,
    "quantization_time_seconds": quant_elapsed,
}
with open(os.path.join(AWQ_OUTPUT_DIR, "quantization_log.json"), "w") as f:
    json.dump(quant_log, f, indent=2, ensure_ascii=False)

total_stage2_time = time.time() - quant_start_time
print(f"\n{'=' * 60}")
print(f"Stage 2 완료! 소요 시간: {total_stage2_time:.1f}초 ({total_stage2_time/60:.1f}분)")
print(f"AWQ 모델 저장 위치: {AWQ_OUTPUT_DIR}")
print(f"{'=' * 60}")

# 메모리 정리
del model
gc.collect()
torch.cuda.empty_cache()
print("\nGPU 메모리 해제 완료")

---
## 6. AWQ 모델 추론 검증

양자화된 모델이 정상적으로 동작하는지 추론 테스트를 수행한다.

In [ ]:
# ============================================================
# AWQ 모델 추론 검증
# ============================================================
print("=" * 60)
print("AWQ 모델 추론 검증")
print("=" * 60)

from transformers import AutoTokenizer, AutoModelForCausalLM

print("\n[1/2] AWQ 모델 로드...")
awq_tokenizer = AutoTokenizer.from_pretrained(AWQ_OUTPUT_DIR, trust_remote_code=True)
awq_model = AutoModelForCausalLM.from_pretrained(
    AWQ_OUTPUT_DIR,
    device_map="auto",
    trust_remote_code=True,
)
awq_model.eval()

mem_awq = torch.cuda.memory_allocated() / 1024**3
print(f"  GPU 메모리 사용: {mem_awq:.2f} GB")

print("\n[2/2] 추론 테스트...")

eos_ids = get_eos_ids(awq_tokenizer)

test_prompts_awq = [
    "주민등록증 재발급 절차가 어떻게 되나요?",
    "우리 동네 도로에 포트홀이 생겨서 위험합니다.",
    "기초생활수급자 신청 방법을 알려주세요.",
]

awq_results = []
for i, prompt in enumerate(test_prompts_awq):
    messages = [
        {"role": "system", "content": "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."},
        {"role": "user", "content": f"다음 민원에 대해 공손하고 명확한 답변을 작성하세요.\n\n민원 내용: {prompt}"},
    ]
    input_ids = awq_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(awq_model.device)

    with torch.no_grad():
        output = awq_model.generate(
            input_ids,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos_ids,
        )

    response = awq_tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=False)
    clean_response = re.sub(r"<thought>.*?</thought>", "", response, flags=re.DOTALL).strip()
    has_eos = any(tok in response for tok in ['[|endofturn|]', '<|endofturn|>'])
    
    print(f"\n  --- 테스트 {i+1} ---")
    print(f"  질의: {prompt}")
    print(f"  응답 (300자): {clean_response[:300]}")
    print(f"  EOS 생성: {'Yes' if has_eos else 'No'}")
    
    awq_results.append({
        "prompt": prompt,
        "response": clean_response[:500],
        "has_eos": has_eos,
    })

wandb.log({
    "awq_gpu_mem_gb": mem_awq,
    "awq_inference_results": awq_results,
})

print(f"\n{'=' * 60}")
print("AWQ 모델 추론 검증 완료")
print(f"{'=' * 60}")

# 메모리 정리
del awq_model
gc.collect()
torch.cuda.empty_cache()

---
## 7. HuggingFace Hub 업로드

머지 모델과 AWQ 모델을 HuggingFace Hub에 업로드한다.

In [ ]:
# ============================================================
# HuggingFace Hub 업로드
# ============================================================
from huggingface_hub import HfApi, login

print("=" * 60)
print("HuggingFace Hub 업로드")
print("=" * 60)

# HuggingFace 로그인
login(token=os.environ["HF_TOKEN"])
api = HfApi()

# ── 머지 모델 업로드 ──
print(f"\n[1/2] 머지 모델 업로드: {HF_MERGED_REPO}")
print(f"  소스: {MERGED_OUTPUT_DIR}")
print(f"  예상 크기: {merged_size_gb:.2f} GB (시간이 다소 걸릴 수 있음)")

try:
    api.create_repo(HF_MERGED_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=MERGED_OUTPUT_DIR,
        repo_id=HF_MERGED_REPO,
        commit_message="Upload GovOn-EXAONE-Merged-v2 (BF16)",
    )
    print(f"  [OK] 머지 모델 업로드 완료: https://huggingface.co/{HF_MERGED_REPO}")
    wandb.log({"hf_merged_repo": HF_MERGED_REPO, "hf_merged_upload": True})
except Exception as e:
    print(f"  [ERROR] 머지 모델 업로드 실패: {e}")
    wandb.log({"hf_merged_upload": False, "hf_merged_error": str(e)})

# ── AWQ 모델 업로드 ──
print(f"\n[2/2] AWQ 모델 업로드: {HF_AWQ_REPO}")
print(f"  소스: {AWQ_OUTPUT_DIR}")
print(f"  예상 크기: {awq_size_gb:.2f} GB")

try:
    api.create_repo(HF_AWQ_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=AWQ_OUTPUT_DIR,
        repo_id=HF_AWQ_REPO,
        commit_message="Upload GovOn-EXAONE-AWQ-v2 (W4A16g128)",
    )
    print(f"  [OK] AWQ 모델 업로드 완료: https://huggingface.co/{HF_AWQ_REPO}")
    wandb.log({"hf_awq_repo": HF_AWQ_REPO, "hf_awq_upload": True})
except Exception as e:
    print(f"  [ERROR] AWQ 모델 업로드 실패: {e}")
    wandb.log({"hf_awq_upload": False, "hf_awq_error": str(e)})

print(f"\n{'=' * 60}")
print("HuggingFace Hub 업로드 완료")
print(f"{'=' * 60}")

---
## 8. 최종 요약 및 WandB 완료

In [ ]:
# ============================================================
# 최종 요약 및 WandB 완료
# ============================================================
total_elapsed = time.time() - merge_start_time

print("=" * 60)
print("파이프라인 완료 요약")
print("=" * 60)
print(f"")
print(f"  베이스 모델:       {BASE_MODEL_ID}")
print(f"  LoRA 어댑터:       {ADAPTER_ID}")
print(f"  EXAONE 리비전:     {EXAONE_REVISION}")
print(f"")
print(f"  머지 모델 크기:    {merged_size_gb:.2f} GB (BF16)")
print(f"  AWQ 모델 크기:     {awq_size_gb:.2f} GB (W4A16g128)")
print(f"  압축률:            {compression_ratio:.2f}x")
print(f"  크기 감소:         {size_reduction_pct:.1f}%")
print(f"")
print(f"  머지 소요 시간:    {merge_elapsed:.1f}초 ({merge_elapsed/60:.1f}분)")
print(f"  양자화 소요 시간:  {quant_elapsed:.1f}초 ({quant_elapsed/60:.1f}분)")
print(f"  전체 소요 시간:    {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")
print(f"")
print(f"  HF 머지 모델:      https://huggingface.co/{HF_MERGED_REPO}")
print(f"  HF AWQ 모델:       https://huggingface.co/{HF_AWQ_REPO}")
print(f"")
print(f"{'=' * 60}")

# WandB 최종 로깅
wandb.log({
    "total_pipeline_time_seconds": total_elapsed,
    "total_pipeline_time_minutes": total_elapsed / 60,
})

# WandB summary에 주요 지표 기록
wandb.summary["merged_model_size_gb"] = merged_size_gb
wandb.summary["awq_model_size_gb"] = awq_size_gb
wandb.summary["compression_ratio"] = compression_ratio
wandb.summary["size_reduction_pct"] = size_reduction_pct
wandb.summary["total_time_minutes"] = total_elapsed / 60
wandb.summary["hf_merged_repo"] = HF_MERGED_REPO
wandb.summary["hf_awq_repo"] = HF_AWQ_REPO

wandb.finish()
print("\nWandB 로깅 완료")